# Benchmark trung thực: quantum ML có thắng classical không?

Bạn đóng vai AI Engineer được CTO hỏi: "Một vendor chào giải pháp quantum machine learning cho bài toán phân loại của
mình. Có nên đầu tư không?" Bạn trả lời bằng **số đo**, không bằng cảm giác: barren plateau đo được, một kernel lượng tử
"dequantize" được thành công thức cổ điển, một benchmark có baseline mạnh và khoảng tin cậy, và tác động của nhiễu cộng
shots. Kết quả cuối là `MEMO.md` một trang rưỡi cho người không đọc code.

**Cách làm bài**
- Ba hàm được chấm bằng test (cell có `# TODO`): `angle_kernel_classical`, `gradient_variance`, `paired_ci`.
- Phần còn lại là thí nghiệm của bạn trong cell `explore` (dòng đầu `# explore: ...`), thêm bao nhiêu cũng được.
  Được dùng PennyLane, numpy (simulator bài Basic 01), scikit-learn, scipy.
- Chấm: `uv run pytest tracks/ai-engineer/modules/22-quantum-ml/exercises/advanced/01-honest-benchmark -q`; nội dung `MEMO.md` chấm theo rubric trong `SOLUTION.md`.

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import numpy
import time
import warnings

import numpy as np
import pennylane as qml
from scipy import stats

warnings.filterwarnings("ignore", category=UserWarning)

## 1. Barren plateau đo bằng số

Ansatz "hardware-efficient" $$n$$ qubit, $$L$$ layer, trạng thái đầu $$|0\dots0\rangle$$, tham số `theta` shape
`(L, n, 2)`. Mỗi layer $$l$$: với mỗi wire $$w$$, `RY(theta[l, w, 0])` rồi `RZ(theta[l, w, 1])`; sau đó `CZ` trên từng cặp
$$(w, w+1)$$, $$w = 0..n-2$$. Hai hàm chi phí:

- `"global"`: $$C = \langle Z_0 Z_1 \cdots Z_{n-1} \rangle$$ (quan sát trên mọi qubit),
- `"local"`: $$C = \langle Z_0 \rangle$$.

`gradient_variance(n_qubits, n_layers, cost, n_samples=200, seed=0)`: rút **một lần**
`theta = np.random.default_rng(seed).uniform(0, 2 * np.pi, size=(n_samples, n_layers, n_qubits, 2))`, với mỗi mẫu tính
$$\partial C / \partial \theta_{0,0,0}$$ (RY đầu tiên trên wire 0) bằng parameter-shift $$s = \pi/2$$, trả **phương sai**
(`np.var`, ddof 0) của `n_samples` đạo hàm đó, kiểu float Python. Với $$n = 10$$, $$L = 20$$ hàm phải chạy dưới vài giây:
batch hoá theo mẫu (simulator numpy của bạn có trục batch, hoặc parameter broadcasting của PennyLane).

In [ ]:
def gradient_variance(n_qubits: int, n_layers: int, cost: str, n_samples: int = 200, seed: int = 0) -> float:
    """Var của dC/dθ[0, 0, 0] trên n_samples bộ tham số ngẫu nhiên U(0, 2π). cost: "global" hoặc "local"."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: TODO bảng phương sai gradient, n = 2, 4, 6, 8, 10; L = 1, 5, 20; global và local.
# Vẽ log2(variance) theo n và ước lượng độ dốc (np.polyfit). Độ dốc -1 nghĩa là mỗi qubit thêm làm phương sai giảm một nửa.

## 2. Dequantization: kernel lượng tử tính được bằng một dòng numpy

Feature map **angle embedding** chỉ quay từng qubit: $$|\varphi(x)\rangle = \bigotimes_w RY(x_w)|0\rangle$$, không có gate hai
qubit nào. Kernel fidelity của nó là tích của các kernel một qubit, và mỗi kernel một qubit có dạng đóng.

`angle_kernel_classical(XA, XB)`: trả ma trận `(len(XA), len(XB))` bằng $$|\langle \varphi(a) | \varphi(b) \rangle|^2$$ của
angle embedding RY, tính **bằng công thức đóng** (tự suy ra từ ma trận RY), không dùng statevector. Rồi trong cell
`explore`: kiểm nó khớp kernel tính bằng PennyLane (`qml.AngleEmbedding(..., rotation="Y")` + `qml.state()`), và đo thời
gian hai cách với 300 điểm, $$d = 4, 8, 12, 16$$ qubit.

In [ ]:
def angle_kernel_classical(XA: np.ndarray, XB: np.ndarray) -> np.ndarray:
    """Kernel fidelity của angle embedding RY, tính bằng công thức đóng: shape (len(XA), len(XB))."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: TODO kiểm angle_kernel_classical khớp PennyLane (sai lệch cỡ 1e-15) và so thời gian theo số qubit.

## 3. Benchmark có baseline mạnh và khoảng tin cậy

`paired_ci(scores_a, scores_b, confidence=0.95)`: hai mảng điểm của hai model trên **cùng** các fold. Với
$$d_k = a_k - b_k$$ ($$K$$ fold), trả dict `{"mean_diff", "std_diff", "ci_low", "ci_high"}`: trung bình của $$d$$, độ lệch
chuẩn mẫu (ddof 1), và khoảng tin cậy Student-t $$\bar d \pm t_{(1+c)/2,\,K-1}\cdot \mathrm{std}/\sqrt{K}$$ (`stats.t.ppf`),
mọi giá trị là float Python.

Rồi chạy benchmark, tối thiểu:
- **Dữ liệu**: Iris (4 qubit), moons 300 điểm `noise=0.2, random_state=0` (2 qubit), breast cancer qua PCA 4 thành
  phần (4 qubit). 5-fold stratified `random_state=0`, mọi tiền xử lý fit trong fold train.
- **Model**: SVM với kernel ZZ (feature map của bài Intermediate 02), SVM với `angle_kernel_classical`, SVM RBF,
  logistic regression. **Cùng ngân sách tuning**: mỗi model chọn một siêu tham số trên fold train (scale của kernel
  lượng tử bằng alignment hay inner CV, gamma của RBF bằng inner CV) từ lưới cỡ như nhau.
- **Báo cáo**: accuracy mean ± std, thời gian, và `paired_ci` của ZZ so với SVM RBF và so với kernel angle (ablation bỏ
  entanglement).

In [ ]:
def paired_ci(scores_a, scores_b, confidence: float = 0.95) -> dict[str, float]:
    """Khoảng tin cậy Student-t cho trung bình của a − b trên cùng các fold."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: TODO benchmark 3 bộ dữ liệu x 4 model, 5-fold, cùng ngân sách tuning; in bảng và paired_ci.

## 4. Nhiễu và shots

Mô phỏng phần cứng NISQ cho kernel ZZ trên moons (lấy 150 điểm đầu của `make_moons(150, noise=0.2, random_state=0)`
cho nhanh, `StandardScaler` fit trên fold train rồi nhân scale 0.3, 5-fold như mục 3): mạch compute-uncompute $$U(b)^\dagger U(a)$$ trên `qml.device("default.mixed", wires=2)`, thêm
`qml.DepolarizingChannel(p)` trên các wire vừa có gate sau mỗi khối của feature map, đọc xác suất ra $$|00\rangle$$. So $$p \in \{0, 0.05\}$$ có và không có shots
(100, 30). Đo cả **độ tương phản** của kernel (trung bình đường chéo trừ trung bình ngoài đường chéo), không chỉ accuracy.
`default.mixed` chậm hơn statevector nhiều: một giá trị $$p$$ khoảng 30 giây.

In [ ]:
# explore: TODO kernel ZZ có nhiễu depolarizing + shots; bảng accuracy, đường chéo, ngoài đường chéo.

## 5. Memo

Viết `MEMO.md` (cùng thư mục) theo khung có sẵn, rồi làm phần trắc nghiệm của bài trong app.